# HAM10000 Skin Disease Classification — Evaluation Notebook

This notebook provides a **clean, decoupled evaluation** of all trained models
using the pre-saved `data_splits/test_split.csv` file produced by `src/prepare_data.py`.

## Contents
1. Setup & Imports
2. Configuration
3. Test Dataset & DataLoader
4. Per-Model Evaluation (EfficientNet-B0, ResNet34, MobileNetV3-Large)
5. ResNet34 + XGBoost Evaluation
6. Soft-Voting Ensemble
7. Comparative Results Summary
8. Confusion Matrices
9. Per-Class Performance Bar Chart

In [ ]:
# ─── 0. Install / upgrade packages if running on Kaggle ──────────────────────
# Uncomment the next line on a fresh Kaggle environment:
# !pip install -q albumentations==1.4.11 imbalanced-learn xgboost

In [ ]:
# ─── 1. Setup & Imports ───────────────────────────────────────────────────────
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm
from xgboost import XGBClassifier

# Add project root to path
PROJECT_ROOT = Path(".").resolve().parent  # one level up from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import HAM10000Dataset, get_transforms, NUM_CLASSES, CLASS_NAMES
from src.train_resnet import get_backbone

print(f"Project root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {device}")

In [ ]:
# ─── 2. Configuration ─────────────────────────────────────────────────────────
CFG = {
    "test_csv":          PROJECT_ROOT / "data_splits" / "test_split.csv",
    "efficientnet_ckpt": PROJECT_ROOT / "checkpoints" / "efficientnet_best.pth",
    "resnet_ckpt":       PROJECT_ROOT / "checkpoints" / "resnet_best.pth",
    "mobilenet_ckpt":    PROJECT_ROOT / "checkpoints" / "mobilenet_best.pth",
    "xgb_model":         PROJECT_ROOT / "checkpoints" / "resnet_xgboost.json",
    "batch_size":        32,
    "num_workers":       2,
}
for k, v in CFG.items():
    print(f"  {k:22s}: {v}")

In [ ]:
# ─── 3. Test Dataset & DataLoader ─────────────────────────────────────────────
test_ds = HAM10000Dataset(
    CFG["test_csv"],
    transform=get_transforms("test"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
)
print(f"Test samples : {len(test_ds)}")
print(f"Batches      : {len(test_loader)}")

In [ ]:
# ─── Helper: model loaders ────────────────────────────────────────────────────

def load_efficientnet(ckpt_path, device):
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, NUM_CLASSES),
    )
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    return model.to(device).eval()


def load_resnet(ckpt_path, device):
    model = models.resnet34(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, NUM_CLASSES),
    )
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    return model.to(device).eval()


def load_mobilenet(ckpt_path, device):
    model = models.mobilenet_v3_large(weights=None)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, NUM_CLASSES)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    return model.to(device).eval()


@torch.no_grad()
def get_probs_and_labels(model, loader, device):
    """Returns (probs [N,7], true_labels [N])."""
    all_probs, all_labels = [], []
    for imgs, lbls in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        logits = model(imgs)
        probs  = F.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(lbls.numpy())
    return np.vstack(all_probs), np.concatenate(all_labels)


print("Helper functions defined.")

In [ ]:
# ─── 4. Per-Model Evaluation ──────────────────────────────────────────────────
model_results = {}  # name → {probs, preds, acc, report}

model_configs = [
    ("EfficientNet-B0", CFG["efficientnet_ckpt"], load_efficientnet),
    ("ResNet34",        CFG["resnet_ckpt"],        load_resnet),
    ("MobileNetV3",     CFG["mobilenet_ckpt"],     load_mobilenet),
]

true_labels = None

for model_name, ckpt_path, loader_fn in model_configs:
    print(f"\n{'─'*60}")
    print(f"Evaluating: {model_name}")
    if not Path(ckpt_path).exists():
        print(f"  ⚠  Checkpoint not found: {ckpt_path}  — skipping")
        continue

    model = loader_fn(ckpt_path, device)
    probs, labels = get_probs_and_labels(model, test_loader, device)
    del model
    torch.cuda.empty_cache()

    if true_labels is None:
        true_labels = labels

    preds = probs.argmax(axis=1)
    acc   = accuracy_score(labels, preds)

    report = classification_report(
        labels, preds,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True,
    )
    model_results[model_name] = {
        "probs":  probs,
        "preds":  preds,
        "acc":    acc,
        "report": report,
    }

    print(f"  Accuracy : {acc:.4f}")
    print(classification_report(labels, preds, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# ─── 5. ResNet34 + XGBoost Evaluation ─────────────────────────────────────────
print("\n" + "─"*60)
print("Evaluating: ResNet34 + XGBoost")

if Path(CFG["resnet_ckpt"]).exists() and Path(CFG["xgb_model"]).exists():
    # Extract features using the frozen backbone
    backbone = get_backbone(str(CFG["resnet_ckpt"]), device)

    @torch.no_grad()
    def extract_feats(backbone, loader, device):
        feats, lbls = [], []
        for imgs, labels in tqdm(loader, leave=False, desc="Extracting"):
            imgs = imgs.to(device)
            f = backbone(imgs).squeeze(-1).squeeze(-1)
            feats.append(f.cpu().numpy())
            lbls.append(labels.numpy())
        return np.vstack(feats), np.concatenate(lbls)

    X_test, y_test = extract_feats(backbone, test_loader, device)
    del backbone

    xgb = XGBClassifier()
    xgb.load_model(str(CFG["xgb_model"]))
    xgb_preds = xgb.predict(X_test)
    xgb_acc   = accuracy_score(y_test, xgb_preds)

    xgb_report = classification_report(
        y_test, xgb_preds,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True,
    )
    model_results["ResNet34+XGBoost"] = {
        "probs":  None,  # XGBoost doesn't contribute to DL ensemble
        "preds":  xgb_preds,
        "acc":    xgb_acc,
        "report": xgb_report,
    }

    print(f"  Accuracy : {xgb_acc:.4f}")
    print(classification_report(y_test, xgb_preds, target_names=CLASS_NAMES, zero_division=0))
else:
    print("  ⚠  ResNet or XGBoost checkpoint missing — skipping")

In [ ]:
# ─── 6. Soft-Voting Ensemble (DL models only) ─────────────────────────────────
print("\n" + "─"*60)
print("Soft-Voting Ensemble")

dl_probs = [
    v["probs"]
    for k, v in model_results.items()
    if v["probs"] is not None
]

if dl_probs:
    ensemble_probs = np.mean(np.stack(dl_probs, axis=0), axis=0)
    ensemble_preds = ensemble_probs.argmax(axis=1)
    ensemble_acc   = accuracy_score(true_labels, ensemble_preds)

    ensemble_report = classification_report(
        true_labels, ensemble_preds,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True,
    )
    model_results["Ensemble (Soft-Voting)"] = {
        "probs":  ensemble_probs,
        "preds":  ensemble_preds,
        "acc":    ensemble_acc,
        "report": ensemble_report,
    }

    print(f"  Ensemble Accuracy : {ensemble_acc:.4f}")
    print(classification_report(true_labels, ensemble_preds, target_names=CLASS_NAMES, zero_division=0))
else:
    print("  ⚠  No DL model results available for ensemble")

In [ ]:
# ─── 7. Comparative Results Summary ──────────────────────────────────────────
summary_rows = []
for name, res in model_results.items():
    rep = res["report"]
    row = {
        "Model":          name,
        "Accuracy":       f"{res['acc']:.4f}",
        "Macro-Precision": f"{rep.get('macro avg', {}).get('precision', 0):.4f}",
        "Macro-Recall":   f"{rep.get('macro avg', {}).get('recall', 0):.4f}",
        "Macro-F1":       f"{rep.get('macro avg', {}).get('f1-score', 0):.4f}",
        "Weighted-F1":    f"{rep.get('weighted avg', {}).get('f1-score', 0):.4f}",
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("\n" + "─"*60)
print("Model Comparison Summary")
print(summary_df.to_string(index=False))
summary_df

In [ ]:
# ─── 8. Confusion Matrices ────────────────────────────────────────────────────
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
    )
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True",      fontsize=11)
    ax.set_title(title,        fontsize=13, fontweight="bold")
    plt.xticks(rotation=40, ha="right")
    plt.tight_layout()
    plt.show()

for name, res in model_results.items():
    plot_cm(true_labels if name != "ResNet34+XGBoost" else y_test,
            res["preds"],
            f"Confusion Matrix — {name}")

In [ ]:
# ─── 9. Per-Class F1 Comparison Bar Chart ────────────────────────────────────
# Exclude XGBoost from this chart if desired, or include all
chart_models = [n for n in model_results if model_results[n]["probs"] is not None]

if chart_models:
    x = np.arange(len(CLASS_NAMES))
    width = 0.8 / len(chart_models)

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, name in enumerate(chart_models):
        f1_scores = [
            model_results[name]["report"].get(cls, {}).get("f1-score", 0)
            for cls in CLASS_NAMES
        ]
        ax.bar(x + i * width, f1_scores, width=width, label=name, alpha=0.85)

    ax.set_xticks(x + width * (len(chart_models) - 1) / 2)
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
    ax.set_ylabel("F1-Score", fontsize=11)
    ax.set_title("Per-Class F1-Score Comparison", fontsize=13, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.legend(loc="lower right")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()